# 02 - Limpieza de datos

Cargar los CSV crudos de `data/raw/`, los unifica, limpia del texto, detexta idioma y exporta dos a `data/processed/`

## 1. Imports y configuración

In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
from   langdetect import detect, LangDetectException
import glob
import os

pd.set_option('display.max_colwidth',150)  # para ver texto completo al isnpeccionar

# 2. Cargar y unificar los CSV de todos los restaurantes

In [2]:
#Listado de todos los archivos en data/raw
archivos = glob.glob('../data/raw/*.csv')
print(f'Archivos encontrados: {len(archivos)}')

for a in archivos:
    print(f' -', a)

Archivos encontrados: 8
 - ../data/raw\casa_res_reviews_raw.csv
 - ../data/raw\don_parrilla_steak_house.csv
 - ../data/raw\el_dorado_reviews_raw.csv
 - ../data/raw\la_casa_del_Tomahawk_reviews_raw.csv
 - ../data/raw\la_parrilla_del_ñato_reviews_raw.csv
 - ../data/raw\morogrill_la_terraza_reviews_raw.csv
 - ../data/raw\parrilla_punta_del_este_reviews_raw.csv
 - ../data/raw\rukito_reviews_raw.csv


In [ ]:
# Carga de achivos a df
columnas_utiles = [
    'text', 'stars', 'publishedAtDate', 'title',
    'reviewDetailedRating/Comida', 'reviewDetailedRating/Servicio', 
    'reviewDetailedRating/Ambiente', 'reviewContext/Tiempo de espera',
    'likesCount', 'reviewerNumberOfReviews', 'isLocalGuide'
]

dfs = []
for archivo in archivos: 
    df_temp = pd.read_csv(archivo)
    df_temp = df_temp[columnas_utiles] #Nos quedamos con las columnas útiles
    dfs.append(df_temp)
    
df =pd.concat(dfs, ignore_index=True)
print(f'Total de reseñas combinadas: {len(df)}')
df.head()

Total de reseñas combinadas: 5852


,text,stars,publishedAtDate,title,reviewDetailedRating/Comida,reviewDetailedRating/Servicio,reviewDetailedRating/Ambiente,reviewContext/Tiempo de espera,likesCount,reviewerNumberOfReviews,isLocalGuide
0,"Casa Res.\nUn hermoso restaurante con una vista maravillosa, una comida exquisita, una gran atención al cliente, platos completamente recién hecho...",5,2026-03-03T03:04:39.746Z,Casa Res | Steak House (Mall del Sol),5.0,5.0,5.0,Sin espera,0,5,False
1,"El concepto de Casa Res me fascina, en general mantienen todo su estilo en todos sus locales.\n\nEl espacio del local grande, calculo que ingresan...",4,2025-03-04T06:59:59.379Z,Casa Res | Steak House (Mall del Sol),4.0,3.0,4.0,NaN,0,89,True
2,"Una experiencia terrible, me dijeron que era un buen restaurante y resultó se lo contrario.\nEl servicio lento e ineficiente, no te prestan atenci...",1,2024-10-24T18:46:13.385Z,Casa Res | Steak House (Mall del Sol),1.0,1.0,3.0,NaN,1,114,True
3,Realmente delicioso!! Además se puede elegir el tipo de ensalada de tu preferencia sin costo adicional.,5,2026-06-04T20:18:03.173Z,Casa Res | Steak House (Mall del Sol),NaN,NaN,NaN,De 10 a 30 min,0,7,True
4,"Buena atención, el local es más pequeño que el de San Marino, lo destacable es que el día de la madre nos dieron una mesa con una corta espera, el...",4,2026-05-17T00:33:06.375Z,Casa Res | Steak House (Mall del Sol),5.0,5.0,4.0,NaN,0,234,True


In [4]:
print(f'shape:  {df.shape}')
print("\nRestaurantes cargados")
print(df['title'].value_counts())
print("\nNulos por columna")
print(df.isnull().sum())


shape:  (5852, 11)

Restaurantes cargados
title
La Casa del Tomahawk                        1000
La Parrilla Del Ñato - Urdesa               1000
Rukito Grill&Drink - Alborada               1000
Parrillada Restaurant El Dorado Sauces 3     740
Casa Res | Steak House (Mall del Sol)        672
Parrillada Punta Del Este                    596
MoroGrill - C.C. Las Terrazas                572
Don Parrilla Steak House - Urdesa            272
Name: count, dtype: int64

Nulos por columna
text                               577
stars                                0
publishedAtDate                      0
title                                0
reviewDetailedRating/Comida       4237
reviewDetailedRating/Servicio     4228
reviewDetailedRating/Ambiente     4238
reviewContext/Tiempo de espera    5608
likesCount                           0
reviewerNumberOfReviews              0
isLocalGuide                         0
dtype: int64


# 3. Filtro de fecha (2020 - 2026) - Parsear fechas

In [12]:
# Nueva columna fecha
df['fecha'] = pd.to_datetime(df['publishedAtDate'])

#Extraer año
df['Year'] = df['fecha'].dt.year 

# Filtramos por el periodo 2020 - 2026
df = df[df['Year'] >= 2020].copy()

# Parsear fechas
print(f'Reseñas desde: {df['fecha'].min()}')
print(f'Reseñas desde: {df['fecha'].max()}')
print("Fechas nulas después de to_datetime:", df['publishedAtDate'].isna().sum())

Reseñas desde: 2020-01-01 03:57:22.050000+00:00
Reseñas desde: 2026-08-11 15:12:00.114000+00:00
Fechas nulas después de to_datetime: 0


# 4. Separamos las reviews con texto y de las que no

In [6]:
# Separamos las reseñas con texto vs solo rating
df['tiene_texto'] = df['text'].notna()

print(f'Con texto: {df['tiene_texto'].sum()}')
print(f'Solo rating: {(~df['tiene_texto']).sum()}')

# Dataset compelto (para métricas de rating general)
df_completo = df.copy()

#Dataset solo con texto (para NLP -sentimiento y temas)
df_texto = df[df['tiene_texto']].copy()

Con texto: 3537
Solo rating: 516


# 5. Limpieza de texto y detección de idioma

In [7]:
def limpiar_texto(texto): 
    if pd.isna(texto): 
        return texto
    
    # Normalizar unicode (maneja tildes/ñ correctamente)
    texto = unicodedata.normalize('NFC',texto)
    
    #Quitar saltos de líneas múltiples y espacios extra
    texto = re.sub(r'\s+',' ',texto)
    
    #Quitar espacioes al inicio/final
    texto = texto.strip()
    
    return texto

df_texto['text_limpio'] = df_texto['text'].apply(limpiar_texto)

In [8]:
#Detección de idioma
def detectar_idioma_seguro(texto): 
    if pd.isna(texto) or len(texto.strip()) < 20: 
        return 'es'
    try: 
        return detect(texto)
    except LangDetectException:
        return 'desconocido'
    
df_texto['idioma_detectado'] = df_texto['text_limpio'].apply(detectar_idioma_seguro)
print(df_texto['idioma_detectado'].value_counts())

idioma_detectado
es             3423
pt               50
en               36
it               11
ca                6
ro                3
cy                2
so                2
fr                1
vi                1
desconocido       1
id                1
Name: count, dtype: int64


# 6. Cambio de nombre a columnas

In [9]:
df_texto = df_texto.rename(columns={
    'title': 'restaurante',
    'text_limpio': 'review_texto',
    'stars': 'rating',
    'reviewDetailedRating/Comida':'rating_comida',
    'reviewDetailedRating/Servicio':'rating_servicio',
    'reviewDetailedRating/Ambiente':'rating_ambiente',
    'reviewContext/Tiempo de espera':'tiempo_espera_reportado',
})

df_texto.head(5)

,text,rating,publishedAtDate,restaurante,rating_comida,rating_servicio,rating_ambiente,tiempo_espera_reportado,likesCount,reviewerNumberOfReviews,isLocalGuide,fecha,Year,tiene_texto,review_texto,idioma_detectado
0,"Casa Res.\nUn hermoso restaurante con una vista maravillosa, una comida exquisita, una gran atención al cliente, platos completamente recién hecho...",5,2026-03-03T03:04:39.746Z,Casa Res | Steak House (Mall del Sol),5.0,5.0,5.0,Sin espera,0,5,False,2026-03-03 03:04:39.746000+00:00,2026,True,"Casa Res. Un hermoso restaurante con una vista maravillosa, una comida exquisita, una gran atención al cliente, platos completamente recién hechos...",es
1,"El concepto de Casa Res me fascina, en general mantienen todo su estilo en todos sus locales.\n\nEl espacio del local grande, calculo que ingresan...",4,2025-03-04T06:59:59.379Z,Casa Res | Steak House (Mall del Sol),4.0,3.0,4.0,NaN,0,89,True,2025-03-04 06:59:59.379000+00:00,2025,True,"El concepto de Casa Res me fascina, en general mantienen todo su estilo en todos sus locales. El espacio del local grande, calculo que ingresan má...",es
2,"Una experiencia terrible, me dijeron que era un buen restaurante y resultó se lo contrario.\nEl servicio lento e ineficiente, no te prestan atenci...",1,2024-10-24T18:46:13.385Z,Casa Res | Steak House (Mall del Sol),1.0,1.0,3.0,NaN,1,114,True,2024-10-24 18:46:13.385000+00:00,2024,True,"Una experiencia terrible, me dijeron que era un buen restaurante y resultó se lo contrario. El servicio lento e ineficiente, no te prestan atenció...",es
3,Realmente delicioso!! Además se puede elegir el tipo de ensalada de tu preferencia sin costo adicional.,5,2026-06-04T20:18:03.173Z,Casa Res | Steak House (Mall del Sol),NaN,NaN,NaN,De 10 a 30 min,0,7,True,2026-06-04 20:18:03.173000+00:00,2026,True,Realmente delicioso!! Además se puede elegir el tipo de ensalada de tu preferencia sin costo adicional.,es
4,"Buena atención, el local es más pequeño que el de San Marino, lo destacable es que el día de la madre nos dieron una mesa con una corta espera, el...",4,2026-05-17T00:33:06.375Z,Casa Res | Steak House (Mall del Sol),5.0,5.0,4.0,NaN,0,234,True,2026-05-17 00:33:06.375000+00:00,2026,True,"Buena atención, el local es más pequeño que el de San Marino, lo destacable es que el día de la madre nos dieron una mesa con una corta espera, el...",es


# 7. Exportar DT a csv file

In [10]:
df_completo.to_csv('../data/processed/reviews_all.csv', index=False)
df_texto.to_csv('../data/processed/reviews_with_text.csv',index=False)

print('Archivos Guardados en data/processed/')

Archivos Guardados en data/processed/
